<a href="https://colab.research.google.com/github/MitraShabani/Merge-Order-Bias/blob/main/merge_order_bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai tiktoken

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI()
MODEL_NAME = "gpt-4o-mini"

import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text):
    return len(encoding.encode(text))

print("OpenAI client ready.")

In [ ]:
# Generate text from a prompt
def generate(prompt, max_new_tokens=300):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load
BOOKS_DIR = "/content/drive/MyDrive/merge-order-bias/books"
RESULTS_DIR = "/content/drive/MyDrive/merge-order-bias/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

book_files = [f for f in os.listdir(BOOKS_DIR) if f.endswith(".txt")]
print(f"Found {len(book_files)} books: {book_files}")

In [ ]:
# chunk the book
import re

def split_chapters(full_text):
    if "###CHAPTER###" in full_text:
        chapters = full_text.split("###CHAPTER###")
    else:
        chapters = re.split(r'\n\s*[IVXLCDM]+\s*\n', full_text)
    return [c.strip() for c in chapters if c.strip()]

In [ ]:
# Sub-split any chapter that's too long
MAX_CHUNK_TOKENS = 6000

def split_long_chapter(chapter_text, max_tokens=MAX_CHUNK_TOKENS):
    """Split a chapter into smaller pieces if it exceeds max_tokens, splitting on paragraph breaks."""
    token_count = count_tokens(chapter_text)
    if token_count <= max_tokens:
        return [chapter_text]

    paragraphs = chapter_text.split("\n\n")
    sub_chunks = []
    current = ""
    for para in paragraphs:
        candidate = current + "\n\n" + para if current else para
        if count_tokens(candidate) > max_tokens and current:
            sub_chunks.append(current)
            current = para
        else:
            current = candidate
    if current:
        sub_chunks.append(current)
    return sub_chunks

In [ ]:
# Summarize each chapter

def summarize_chapter(chapter_text):
    """Summarize a chapter, sub-splitting first if it's too long, then combining sub-summaries."""
    pieces = split_long_chapter(chapter_text)
    if len(pieces) == 1:
        prompt = f"""Summarize the following book chapter in a concise paragraph, preserving key plot events, characters, and details:

{pieces[0]}

Summary:"""
        return generate(prompt, max_new_tokens=250)
    else:
        # Summarize each piece, then combine those piece-summaries into one chapter summary
        piece_summaries = []
        for piece in pieces:
            prompt = f"""Summarize the following excerpt in a concise paragraph, preserving key plot events, characters, and details:

{piece}

Summary:"""
            piece_summaries.append(generate(prompt, max_new_tokens=200))
        combine_prompt = f"""Combine the following partial summaries of one book chapter into a single concise chapter summary:

{chr(10).join(piece_summaries)}

Combined summary:"""
        return generate(combine_prompt, max_new_tokens=250)


In [ ]:
# Forward and Backward Merge Pipeline
"""Merge chapter summaries sequentially"""

def merge_prompt(running_summary, new_chunk_summary, chapters_so_far_count, direction):

    if direction == "forward":
      context_note = f"Previous summary (covers chapters 1-{chapters_so_far_count - 1}):"
    else:
      context_note = f"Previous summary (covers {chapters_so_far_count - 1} chapters processed so far, from the end of the book backward):"

    reverse_note = "" if direction == "forward" else (
      "\n5. Note: chapters are being processed out of their original book order "
      "(starting from the end). Just summarize the content given — don't try to "
      "reorder it back into original sequence."
  )

    return f"""You are maintaining a running summary of a book, one chapter at a time. This mirrors how real book-summarization systems work: at each step, you have a summary of everything so far, and you must produce a new summary that reads naturally as ONE continuous narrative — not a list.

    {context_note}
    {running_summary}

    New chapter content to add:
    {new_chunk_summary}

    Your task:
    1. Write a new summary covering all {chapters_so_far_count} chapters processed so far, as flowing prose.
    2. You MUST weave in specific new plot details from the new chapter just given — do not skip or gloss over it.
    3. You are free to compress, shorten, or drop minor detail from previously-processed chapters as needed to fit the new content in — you do not need to preserve every earlier detail verbatim.
    4. Do not simply copy the previous summary and append a sentence — genuinely rewrite it as one coherent narrative.{reverse_note}

    Updated summary:"""

def run_merge(chapter_summaries,direction):
    """Run hierarchical merging over chapter_summaries, either 'forward' or 'backward'.
    Returns (final_summary, merge_log) — merge_log always records ORIGINAL chapter numbers,
    regardless of the order they were processed in, so forward and backward results are
    directly comparable chapter-by-chapter."""
    assert direction in ("forward", "backward")

    if direction == "forward":
        order = list(range(len(chapter_summaries)))  # [0, 1, 2, ..., N-1]
    else:
        order = list(range(len(chapter_summaries) - 1, -1, -1))  # [N-1, ..., 1, 0]

    merge_log = []
    first_idx = order[0]
    running_summary = chapter_summaries[first_idx]
    merge_log.append({"step": 0, "chapters_included": [first_idx + 1], "summary": running_summary})

    chapters_processed = [first_idx + 1]

    for step, idx in enumerate(order[1:], start=1):
        chapters_so_far_count = step + 1
        prompt = merge_prompt(running_summary, chapter_summaries[idx], chapters_so_far_count, direction)
        running_summary = generate(prompt, max_new_tokens=2000)
        chapters_processed.append(idx + 1)
        merge_log.append({"step": step, "chapters_included": sorted(chapters_processed), "summary": running_summary})
        print(f"--- {direction}: after merging chapter {idx + 1} ---")

    return running_summary, merge_log


In [ ]:
# Main batch loop: process every book, both directions
import json

for filename in book_files:
    title = filename.replace(".txt", "").replace("_", " ").title()
    print(f"\n=== Processing: {title} ===")
    book_path = os.path.join(BOOKS_DIR, filename)

    with open(book_path, "r", encoding="utf-8") as f:
        full_text = f.read()

    chapters = split_chapters(full_text)
    print(f"  Found {len(chapters)} chapters.")

    # Pre-check: show how many pieces each chapter will become before spending API calls
    for i, ch in enumerate(chapters):
        pieces = split_long_chapter(ch)
        if len(pieces) > 1:
            print(f"    Chapter {i+1} will be split into {len(pieces)} sub-chunks")
        else:
            print(f"    Chapter {i+1}: OK, no split needed")

    print("  Summarizing chapters...")
    chapter_summaries = [summarize_chapter(ch) for ch in chapters]

    print("  Running forward merge...")
    final_forward, log_forward = run_merge(chapter_summaries, "forward")

    print("  Running backward merge...")
    final_backward, log_backward = run_merge(chapter_summaries, "backward")

    book_slug = filename.replace(".txt", "")

    # Save results
    for direction, final_summary, merge_log in [
        ("forward", final_forward, log_forward),
        ("backward", final_backward, log_backward)
    ]:
        results = {
            "book_title": title,
            "model": MODEL_NAME,
            "merge_order": direction,
            "num_chapters": len(chapters),
            "chapter_summaries": chapter_summaries,
            "merge_log": merge_log,
            "final_summary": final_summary
        }
        out_path = os.path.join(RESULTS_DIR, f"{book_slug}_{direction}.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"  Saved: {out_path}")

print("\n=== Batch complete ===")